In [1]:
# ============================================================
# 02_AURORA_data_preparation_leakage_controlled.ipynb
# AURORA-TWETF Leakage-Controlled Data Preparation
#
# Purpose:
# 1. Load downloaded AURORA panel files from Notebook 01.
# 2. Restrict modeling data to Taiwan trading days.
# 3. Construct leakage-controlled lagged and rolling features.
# 4. Construct 20-day and 60-day future TAIEX ordinal regime labels.
# 5. Save final modeling dataset for later model notebooks.
#
# This notebook DOES NOT train models.
# Model training starts in:
# 03_AURORA_model_zoo_baselines.ipynb
# ============================================================

from __future__ import annotations

import os
import sys
import json
import random
import hashlib
import subprocess
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

def install_if_missing(package_name, import_name=None):
    if import_name is None:
        import_name = package_name
    try:
        return __import__(import_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return __import__(import_name)

pd = install_if_missing("pandas", "pandas")
np = install_if_missing("numpy", "numpy")
pyarrow = install_if_missing("pyarrow", "pyarrow")

import pandas as pd
import numpy as np

# ============================================================
# 1. Reproducibility and paths
# ============================================================

RANDOM_SEED = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

PROJECT_CODE = "AURORA_TWETF"

PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DATA_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
DOCS_DIR = PUBLICATION_ROOT / "docs"

for d in [
    DATA_ROOT,
    PANEL_DATA_DIR,
    MODELING_DIR,
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    DOCS_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Leakage-Controlled Data Preparation")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Project root :", PUBLICATION_ROOT)
print("Panel dir    :", PANEL_DATA_DIR)
print("Modeling dir :", MODELING_DIR)
print("Output root  :", OUTPUT_ROOT)
print("=" * 80)

# ============================================================
# 2. Configuration
# ============================================================

ASSETS = ["0050", "006208", "00692", "00881"]

REQUIRED_TAIWAN_SYMBOLS = ["TAIEX"] + ASSETS

TARGET_COL_20D = "TAIEX_regime_fixed_20d"
TARGET_COL_60D = "TAIEX_regime_fixed_60d"

FUTURE_RETURN_20D_AUDIT = "TAIEX_future_return_20d_audit_only"
FUTURE_RETURN_60D_AUDIT = "TAIEX_future_return_60d_audit_only"

CLASS_LABELS = [0, 1, 2, 3, 4]

REGIME_LABEL_DEFINITION = {
    0: "Strong Bear: future return < -10%",
    1: "Bear: -10% <= future return < -3%",
    2: "Neutral: -3% <= future return <= 3%",
    3: "Bull: 3% < future return <= 10%",
    4: "Strong Bull: future return > 10%",
}

FEATURE_WINDOWS = [5, 10, 20, 60]
RETURN_LAGS = [1, 2, 3, 5, 10, 20]
VOLUME_WINDOWS = [5, 20, 60]

MODEL_DATA_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet"
MODEL_DATA_CSV_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.csv"

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def pct(x, digits=2):
    if pd.isna(x):
        return "NA"
    return f"{100 * x:.{digits}f}%"

def load_panel(name):
    path = PANEL_DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(
            f"Missing panel file: {path}\n"
            "Please run 01_AURORA_download_datasets.ipynb first."
        )
    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    return df

def make_ordinal_target(future_return):
    target = pd.Series(index=future_return.index, dtype="float64")

    target[future_return < -0.10] = 0
    target[(future_return >= -0.10) & (future_return < -0.03)] = 1
    target[(future_return >= -0.03) & (future_return <= 0.03)] = 2
    target[(future_return > 0.03) & (future_return <= 0.10)] = 3
    target[future_return > 0.10] = 4

    return target

def compute_rsi_from_close(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))

    return rsi

def safe_zscore(series, window):
    mu = series.rolling(window).mean()
    sd = series.rolling(window).std()
    return (series - mu) / sd.replace(0.0, np.nan)

def is_feature_name_leakage_prone(col):
    c = str(col).lower()

    leakage_tokens = [
        "future",
        "forward",
        "fwd",
        "lead",
        "ahead",
        "next",
        "t+",
        "target",
        "label",
        "regime",
        "audit_only",
    ]

    for token in leakage_tokens:
        if token in c:
            return True, f"contains_token:{token}"

    raw_ohlcv_exact = [
        "open",
        "high",
        "low",
        "close",
        "adj close",
        "adj_close",
        "volume",
        "ohlcv",
    ]

    if c in raw_ohlcv_exact:
        return True, "raw_ohlcv_exact"

    return False, "kept"

# ============================================================
# 4. Load panel files from Notebook 01
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading panel files")
print("=" * 80)

close_panel = load_panel("AURORA_close_panel.parquet")
open_panel = load_panel("AURORA_open_panel.parquet")
high_panel = load_panel("AURORA_high_panel.parquet")
low_panel = load_panel("AURORA_low_panel.parquet")
volume_panel = load_panel("AURORA_volume_panel.parquet")
return_panel = load_panel("AURORA_return_panel.parquet")
etf_close_panel = load_panel("AURORA_etf_close_panel.parquet")
etf_return_panel = load_panel("AURORA_etf_return_panel.parquet")

print("Close panel shape :", close_panel.shape)
print("Volume panel shape:", volume_panel.shape)
print("Return panel shape:", return_panel.shape)
print("ETF close shape   :", etf_close_panel.shape)
print("ETF return shape  :", etf_return_panel.shape)
print("Close panel date range:", close_panel.index.min().date(), "to", close_panel.index.max().date())

# ============================================================
# 5. Define Taiwan modeling calendar
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Defining Taiwan modeling calendar")
print("=" * 80)

missing_required = [s for s in REQUIRED_TAIWAN_SYMBOLS if s not in close_panel.columns]

if missing_required:
    raise KeyError(
        "Missing required Taiwan symbols in close panel: "
        + ", ".join(missing_required)
    )

taiwan_calendar_mask = close_panel[REQUIRED_TAIWAN_SYMBOLS].notna().all(axis=1)
taiwan_calendar = close_panel.index[taiwan_calendar_mask]

if len(taiwan_calendar) == 0:
    raise RuntimeError("Taiwan modeling calendar is empty.")

print("Taiwan modeling calendar rows:", len(taiwan_calendar))
print("Taiwan modeling date range:", taiwan_calendar.min().date(), "to", taiwan_calendar.max().date())

calendar_report = pd.DataFrame([{
    "timestamp_utc": RUN_TIMESTAMP,
    "close_panel_rows": int(close_panel.shape[0]),
    "close_panel_cols": int(close_panel.shape[1]),
    "taiwan_modeling_rows": int(len(taiwan_calendar)),
    "taiwan_modeling_start": taiwan_calendar.min().strftime("%Y-%m-%d"),
    "taiwan_modeling_end": taiwan_calendar.max().strftime("%Y-%m-%d"),
    "required_symbols": ", ".join(REQUIRED_TAIWAN_SYMBOLS),
}])

calendar_report_path = TABLE_DIR / "table_04_modeling_calendar_report.csv"
calendar_report.to_csv(calendar_report_path, index=False, encoding="utf-8-sig")

# ============================================================
# 6. Align panels to Taiwan trading days
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Aligning panels to Taiwan trading days")
print("=" * 80)

# Conservative alignment:
# 1. Use Taiwan trading dates as the modeling index.
# 2. Forward-fill cross-market prices onto Taiwan dates.
# 3. Every feature is shifted by at least one Taiwan trading day.
#    This prevents same-day cross-market leakage.

aligned_close = close_panel.reindex(taiwan_calendar).ffill()
aligned_open = open_panel.reindex(taiwan_calendar).ffill()
aligned_high = high_panel.reindex(taiwan_calendar).ffill()
aligned_low = low_panel.reindex(taiwan_calendar).ffill()
aligned_volume = volume_panel.reindex(taiwan_calendar).ffill()

aligned_return = aligned_close.pct_change().replace([np.inf, -np.inf], np.nan)

alignment_report_rows = []

for symbol in aligned_close.columns:
    alignment_report_rows.append({
        "symbol_name": symbol,
        "n_taiwan_calendar_rows": int(len(taiwan_calendar)),
        "n_nonmissing_close_after_ffill": int(aligned_close[symbol].notna().sum()),
        "missing_close_after_ffill": int(aligned_close[symbol].isna().sum()),
        "missing_rate_after_ffill": float(aligned_close[symbol].isna().mean()),
        "first_valid_date_after_ffill": (
            aligned_close[symbol].dropna().index.min().strftime("%Y-%m-%d")
            if aligned_close[symbol].notna().sum() > 0 else None
        ),
        "last_valid_date_after_ffill": (
            aligned_close[symbol].dropna().index.max().strftime("%Y-%m-%d")
            if aligned_close[symbol].notna().sum() > 0 else None
        ),
    })

alignment_report = pd.DataFrame(alignment_report_rows)
alignment_report_path = TABLE_DIR / "table_05_alignment_report.csv"
alignment_report.to_csv(alignment_report_path, index=False, encoding="utf-8-sig")

print("Aligned close shape:", aligned_close.shape)
print("Aligned return shape:", aligned_return.shape)

# ============================================================
# 7. Construct leakage-controlled features
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Constructing lagged and rolling features")
print("=" * 80)

features = pd.DataFrame(index=taiwan_calendar)

feature_metadata_rows = []

for symbol in aligned_close.columns:
    close = aligned_close[symbol].astype(float)
    daily_ret = aligned_return[symbol].astype(float)

    # Lagged returns
    for lag in RETURN_LAGS:
        col = f"{symbol}_ret_lag_{lag}d"
        features[col] = daily_ret.shift(lag)
        feature_metadata_rows.append({
            "feature": col,
            "source_symbol": symbol,
            "feature_group": "lagged_return",
            "window_or_lag": lag,
            "leakage_control": f"daily return shifted by {lag} Taiwan trading day(s)",
        })

    # Rolling momentum, volatility, moving-average ratio
    for win in FEATURE_WINDOWS:
        col = f"{symbol}_momentum_{win}d_lag1"
        features[col] = close.pct_change(win).shift(1)
        feature_metadata_rows.append({
            "feature": col,
            "source_symbol": symbol,
            "feature_group": "rolling_momentum",
            "window_or_lag": win,
            "leakage_control": "rolling momentum shifted by 1 Taiwan trading day",
        })

        col = f"{symbol}_volatility_{win}d_lag1"
        features[col] = daily_ret.rolling(win).std().shift(1)
        feature_metadata_rows.append({
            "feature": col,
            "source_symbol": symbol,
            "feature_group": "rolling_volatility",
            "window_or_lag": win,
            "leakage_control": "rolling volatility shifted by 1 Taiwan trading day",
        })

        col = f"{symbol}_ma_ratio_{win}d_lag1"
        features[col] = (close / close.rolling(win).mean() - 1.0).shift(1)
        feature_metadata_rows.append({
            "feature": col,
            "source_symbol": symbol,
            "feature_group": "moving_average_ratio",
            "window_or_lag": win,
            "leakage_control": "moving-average ratio shifted by 1 Taiwan trading day",
        })

        col = f"{symbol}_return_zscore_{win}d_lag1"
        features[col] = safe_zscore(daily_ret, win).shift(1)
        feature_metadata_rows.append({
            "feature": col,
            "source_symbol": symbol,
            "feature_group": "return_zscore",
            "window_or_lag": win,
            "leakage_control": "return z-score shifted by 1 Taiwan trading day",
        })

    # RSI
    col = f"{symbol}_rsi_14d_lag1"
    features[col] = compute_rsi_from_close(close, window=14).shift(1)
    feature_metadata_rows.append({
        "feature": col,
        "source_symbol": symbol,
        "feature_group": "rsi",
        "window_or_lag": 14,
        "leakage_control": "RSI shifted by 1 Taiwan trading day",
    })

    # Drawdown
    col = f"{symbol}_drawdown_60d_lag1"
    rolling_peak = close.rolling(60).max()
    features[col] = (close / rolling_peak - 1.0).shift(1)
    feature_metadata_rows.append({
        "feature": col,
        "source_symbol": symbol,
        "feature_group": "rolling_drawdown",
        "window_or_lag": 60,
        "leakage_control": "rolling drawdown shifted by 1 Taiwan trading day",
    })

# Volume features
for symbol in aligned_volume.columns:
    volume = aligned_volume[symbol].astype(float)

    for win in VOLUME_WINDOWS:
        col = f"{symbol}_volume_zscore_{win}d_lag1"
        features[col] = safe_zscore(volume, win).shift(1)
        feature_metadata_rows.append({
            "feature": col,
            "source_symbol": symbol,
            "feature_group": "volume_zscore",
            "window_or_lag": win,
            "leakage_control": "volume z-score shifted by 1 Taiwan trading day",
        })

        col = f"{symbol}_volume_change_{win}d_lag1"
        features[col] = volume.pct_change(win).shift(1)
        feature_metadata_rows.append({
            "feature": col,
            "source_symbol": symbol,
            "feature_group": "volume_change",
            "window_or_lag": win,
            "leakage_control": "volume change shifted by 1 Taiwan trading day",
        })

# Cross-market spread features, all lagged
def add_spread_feature(name, lhs, rhs, window=None):
    if lhs not in aligned_return.columns or rhs not in aligned_return.columns:
        return

    spread = aligned_return[lhs] - aligned_return[rhs]

    if window is None:
        col = f"{name}_lag1"
        features[col] = spread.shift(1)
        win_value = 1
        group = "cross_market_return_spread"
    else:
        col = f"{name}_{window}d_mean_lag1"
        features[col] = spread.rolling(window).mean().shift(1)
        win_value = window
        group = "cross_market_rolling_spread"

    feature_metadata_rows.append({
        "feature": col,
        "source_symbol": f"{lhs}_minus_{rhs}",
        "feature_group": group,
        "window_or_lag": win_value,
        "leakage_control": "cross-market spread shifted by 1 Taiwan trading day",
    })

add_spread_feature("TAIEX_minus_SP500_ret", "TAIEX", "SP500")
add_spread_feature("TAIEX_minus_NASDAQ_ret", "TAIEX", "NASDAQ")
add_spread_feature("SOXX_minus_TAIEX_ret", "SOXX", "TAIEX")
add_spread_feature("NASDAQ_minus_SP500_ret", "NASDAQ", "SP500")
add_spread_feature("KOSPI_minus_TAIEX_ret", "KOSPI", "TAIEX")
add_spread_feature("NIKKEI225_minus_TAIEX_ret", "NIKKEI225", "TAIEX")
add_spread_feature("HANGSENG_minus_TAIEX_ret", "HANGSENG", "TAIEX")

for win in [5, 20]:
    add_spread_feature("TAIEX_minus_SP500_ret", "TAIEX", "SP500", window=win)
    add_spread_feature("SOXX_minus_TAIEX_ret", "SOXX", "TAIEX", window=win)
    add_spread_feature("NASDAQ_minus_SP500_ret", "NASDAQ", "SP500", window=win)

# Remove infinite values
features = features.replace([np.inf, -np.inf], np.nan)

feature_metadata = pd.DataFrame(feature_metadata_rows)

print("Raw generated feature matrix shape:", features.shape)

# ============================================================
# 8. Construct target labels
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Constructing 20-day and 60-day ordinal TAIEX targets")
print("=" * 80)

taiex_close = aligned_close["TAIEX"].astype(float)

future_ret_20d = taiex_close.shift(-20) / taiex_close - 1.0
future_ret_60d = taiex_close.shift(-60) / taiex_close - 1.0

target_20d = make_ordinal_target(future_ret_20d)
target_60d = make_ordinal_target(future_ret_60d)

target_df = pd.DataFrame(index=taiwan_calendar)
target_df[FUTURE_RETURN_20D_AUDIT] = future_ret_20d
target_df[FUTURE_RETURN_60D_AUDIT] = future_ret_60d
target_df[TARGET_COL_20D] = target_20d
target_df[TARGET_COL_60D] = target_60d

# Save target audit separately, but do not include audit-only future returns in modeling matrix.
target_audit_path = MODELING_DIR / "AURORA_TWETF_target_audit_only.parquet"
target_df.to_parquet(target_audit_path)

target_distribution_rows = []

for target_col in [TARGET_COL_20D, TARGET_COL_60D]:
    counts = target_df[target_col].dropna().astype(int).value_counts().sort_index()

    for cls in CLASS_LABELS:
        n = int(counts.get(cls, 0))
        total = int(target_df[target_col].dropna().shape[0])
        share = float(n / total) if total > 0 else np.nan

        target_distribution_rows.append({
            "target_col": target_col,
            "class": cls,
            "regime_name": REGIME_LABEL_DEFINITION[cls],
            "n": n,
            "share": share,
        })

target_distribution = pd.DataFrame(target_distribution_rows)

target_distribution_path = TABLE_DIR / "table_06_target_distribution_20d_60d.csv"
target_distribution.to_csv(target_distribution_path, index=False, encoding="utf-8-sig")

print("Target distribution:")
print(target_distribution.to_string(index=False))

# ============================================================
# 9. Leakage audit
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Leakage audit")
print("=" * 80)

leakage_audit_rows = []

for col in features.columns:
    bad, reason = is_feature_name_leakage_prone(col)

    if bad:
        status = "removed"
    else:
        status = "kept"

    leakage_audit_rows.append({
        "feature": col,
        "status": status,
        "reason": reason,
    })

leakage_audit = pd.DataFrame(leakage_audit_rows)

removed_features = leakage_audit.loc[
    leakage_audit["status"] == "removed",
    "feature",
].tolist()

kept_features = leakage_audit.loc[
    leakage_audit["status"] == "kept",
    "feature",
].tolist()

features_kept = features[kept_features].copy()

# Remove all-null features
all_null_features = [
    c for c in features_kept.columns
    if features_kept[c].notna().sum() == 0
]

if all_null_features:
    features_kept = features_kept.drop(columns=all_null_features)

    leakage_audit.loc[
        leakage_audit["feature"].isin(all_null_features),
        ["status", "reason"]
    ] = ["removed", "all_null_after_alignment"]

kept_features = list(features_kept.columns)

leakage_audit_path = TABLE_DIR / "table_07_leakage_audit.csv"
leakage_audit.to_csv(leakage_audit_path, index=False, encoding="utf-8-sig")

feature_metadata = feature_metadata[feature_metadata["feature"].isin(kept_features)].copy()

print("Generated features:", features.shape[1])
print("Kept features:", len(kept_features))
print("Removed features:", int((leakage_audit["status"] == "removed").sum()))

# ============================================================
# 10. Assemble final modeling dataset
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Assembling final modeling dataset")
print("=" * 80)

modeling_df = features_kept.copy()
modeling_df[TARGET_COL_20D] = target_df[TARGET_COL_20D]
modeling_df[TARGET_COL_60D] = target_df[TARGET_COL_60D]

# Keep rows where the main proposed target exists.
# We use 60-day target as the proposed target because it aligns better with quarterly allocation.
modeling_df = modeling_df.dropna(subset=[TARGET_COL_60D]).copy()

modeling_df[TARGET_COL_20D] = modeling_df[TARGET_COL_20D].astype("Int64")
modeling_df[TARGET_COL_60D] = modeling_df[TARGET_COL_60D].astype(int)

# Drop rows where all features are missing.
feature_nonmissing_count = modeling_df[kept_features].notna().sum(axis=1)
modeling_df = modeling_df.loc[feature_nonmissing_count > 0].copy()

# Sort and ensure no duplicate index.
modeling_df = modeling_df.sort_index()
modeling_df = modeling_df[~modeling_df.index.duplicated(keep="last")]

# Feature missingness report after final row filtering
feature_inventory_rows = []

for col in kept_features:
    s = modeling_df[col]

    feature_inventory_rows.append({
        "feature": col,
        "n_rows": int(modeling_df.shape[0]),
        "missing_count": int(s.isna().sum()),
        "missing_rate": float(s.isna().mean()),
        "nonmissing_count": int(s.notna().sum()),
        "mean": float(s.mean()) if s.notna().sum() > 0 else np.nan,
        "std": float(s.std()) if s.notna().sum() > 1 else np.nan,
        "min": float(s.min()) if s.notna().sum() > 0 else np.nan,
        "max": float(s.max()) if s.notna().sum() > 0 else np.nan,
    })

feature_inventory = pd.DataFrame(feature_inventory_rows)

feature_inventory_path = TABLE_DIR / "table_08_feature_inventory.csv"
feature_metadata_path = TABLE_DIR / "table_09_feature_metadata.csv"
missing_by_date_path = TABLE_DIR / "table_10_modeling_missing_by_date.csv"

feature_inventory.to_csv(feature_inventory_path, index=False, encoding="utf-8-sig")
feature_metadata.to_csv(feature_metadata_path, index=False, encoding="utf-8-sig")

missing_by_date = pd.DataFrame({
    "date": modeling_df.index,
    "n_missing_features": modeling_df[kept_features].isna().sum(axis=1).values,
    "missing_feature_rate": modeling_df[kept_features].isna().mean(axis=1).values,
})

missing_by_date.to_csv(missing_by_date_path, index=False, encoding="utf-8-sig")

# Save final dataset
modeling_df.to_parquet(MODEL_DATA_PATH)
modeling_df.to_csv(MODEL_DATA_CSV_PATH, encoding="utf-8-sig")

print("Final modeling dataset saved:", MODEL_DATA_PATH)
print("Final modeling dataset shape:", modeling_df.shape)
print("Final modeling date range:", modeling_df.index.min().date(), "to", modeling_df.index.max().date())
print("Number of features:", len(kept_features))
print("Target columns:", TARGET_COL_20D, TARGET_COL_60D)

# ============================================================
# 11. Save compact data summary tables
# ============================================================

summary_rows = []

summary_rows.append({
    "item": "project_code",
    "value": PROJECT_CODE,
})

summary_rows.append({
    "item": "timestamp_utc",
    "value": RUN_TIMESTAMP,
})

summary_rows.append({
    "item": "taiwan_modeling_calendar_rows",
    "value": int(len(taiwan_calendar)),
})

summary_rows.append({
    "item": "final_modeling_rows",
    "value": int(modeling_df.shape[0]),
})

summary_rows.append({
    "item": "final_modeling_columns",
    "value": int(modeling_df.shape[1]),
})

summary_rows.append({
    "item": "n_features",
    "value": int(len(kept_features)),
})

summary_rows.append({
    "item": "modeling_start_date",
    "value": modeling_df.index.min().strftime("%Y-%m-%d"),
})

summary_rows.append({
    "item": "modeling_end_date",
    "value": modeling_df.index.max().strftime("%Y-%m-%d"),
})

summary_rows.append({
    "item": "main_target",
    "value": TARGET_COL_60D,
})

summary_rows.append({
    "item": "secondary_target",
    "value": TARGET_COL_20D,
})

data_preparation_summary = pd.DataFrame(summary_rows)

summary_path = TABLE_DIR / "table_11_data_preparation_summary.csv"
data_preparation_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

# ============================================================
# 12. Save configuration and validation report
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "timestamp_utc": RUN_TIMESTAMP,
    "random_seed": RANDOM_SEED,
    "publication_root": str(PUBLICATION_ROOT),
    "input_panel_dir": str(PANEL_DATA_DIR),
    "output_modeling_path": str(MODEL_DATA_PATH),
    "target_audit_path": str(target_audit_path),
    "taiwan_modeling_calendar_rows": int(len(taiwan_calendar)),
    "taiwan_modeling_start": taiwan_calendar.min().strftime("%Y-%m-%d"),
    "taiwan_modeling_end": taiwan_calendar.max().strftime("%Y-%m-%d"),
    "final_modeling_shape": list(modeling_df.shape),
    "final_modeling_start": modeling_df.index.min().strftime("%Y-%m-%d"),
    "final_modeling_end": modeling_df.index.max().strftime("%Y-%m-%d"),
    "n_generated_features": int(features.shape[1]),
    "n_kept_features": int(len(kept_features)),
    "n_removed_features": int((leakage_audit["status"] == "removed").sum()),
    "target_columns": [TARGET_COL_20D, TARGET_COL_60D],
    "main_target_for_AURORA": TARGET_COL_60D,
    "leakage_policy": {
        "modeling_calendar": "Taiwan trading days where TAIEX and all four ETFs are available",
        "cross_market_alignment": "Cross-market prices are forward-filled onto Taiwan dates and all features are shifted by at least one Taiwan trading day",
        "feature_policy": "Only lagged, rolling, or shifted features are retained",
        "excluded_from_modeling_matrix": [
            FUTURE_RETURN_20D_AUDIT,
            FUTURE_RETURN_60D_AUDIT,
            "raw same-day OHLCV variables",
            "future returns",
            "target-derived variables",
        ],
    },
    "regime_label_definition": REGIME_LABEL_DEFINITION,
    "feature_windows": FEATURE_WINDOWS,
    "return_lags": RETURN_LAGS,
    "volume_windows": VOLUME_WINDOWS,
}

validation_report_path = REPORT_DIR / "AURORA_data_preparation_validation_report.json"
save_json(validation_report_path, validation_report)

# Manifest for data and outputs
data_manifest = make_file_manifest(DATA_ROOT)
data_manifest_path = DOCS_DIR / "AURORA_data_preparation_file_manifest_SHA256.csv"
data_manifest.to_csv(data_manifest_path, index=False, encoding="utf-8-sig")

# ============================================================
# 13. Console summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF DATA PREPARATION COMPLETE")
print("=" * 80)
print("Timestamp UTC:", RUN_TIMESTAMP)
print("Modeling dataset:", MODEL_DATA_PATH)
print("Modeling CSV    :", MODEL_DATA_CSV_PATH)
print("Target audit    :", target_audit_path)
print("Validation report:", validation_report_path)
print("File manifest   :", data_manifest_path)

print("\nFinal modeling dataset:")
print("Rows:", modeling_df.shape[0])
print("Columns:", modeling_df.shape[1])
print("Features:", len(kept_features))
print("Date range:", modeling_df.index.min().date(), "to", modeling_df.index.max().date())

print("\n20-day target distribution:")
print(
    target_distribution[target_distribution["target_col"] == TARGET_COL_20D]
    .to_string(index=False)
)

print("\n60-day target distribution:")
print(
    target_distribution[target_distribution["target_col"] == TARGET_COL_60D]
    .to_string(index=False)
)

print("\nSaved tables:")
for p in sorted(TABLE_DIR.glob("table_0*.csv")):
    print(p)

print("\nNext notebook:")
print("03_AURORA_model_zoo_baselines.ipynb")
print("=" * 80)

Mounted at /content/drive
AURORA-TWETF Leakage-Controlled Data Preparation
Timestamp UTC: 2026-06-23T13:58:41Z
Project root : /content/drive/MyDrive/AURORA_TWETF
Panel dir    : /content/drive/MyDrive/AURORA_TWETF/data/panels
Modeling dir : /content/drive/MyDrive/AURORA_TWETF/data/modeling
Output root  : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF

Step 1: Loading panel files
Close panel shape : (1426, 14)
Volume panel shape: (1426, 14)
Return panel shape: (1426, 14)
ETF close shape   : (1426, 4)
ETF return shape  : (1426, 4)
Close panel date range: 2021-01-01 to 2026-06-23

Step 2: Defining Taiwan modeling calendar
Taiwan modeling calendar rows: 1324
Taiwan modeling date range: 2021-01-04 to 2026-06-23

Step 3: Aligning panels to Taiwan trading days
Aligned close shape: (1324, 14)
Aligned return shape: (1324, 14)

Step 4: Constructing lagged and rolling features
Raw generated feature matrix shape: (1324, 433)

Step 5: Constructing 20-day and 60-day ordinal TAIEX targets
Ta